# 🧑‍🏫 Regression Microprojects — Instructor Notebook
*Elements of Data Science* <br>
After Lab 8, each student applies their **regression toolbox** (`standard_units`, `correlation`, `slope`, `intercept`) to a new, real data set with much less scaffolding. This notebook:

1. Reads the **class list** and **randomly assigns** each student to one of five microprojects
2. Saves the assignments to `shared-public` so each student's notebook can confirm their project with `my_project(name)`
3. Shows a live **gallery** of the figures students submit with `submit_plot(...)`, plus a table of their numbers for class discussion

|Project|Data|Twist|Typical *r*|
|---|---|---|---|
|MP1 🌍 CO₂ at Mauna Loa|NOAA annual means, 1959–2025|Errors curve: the rise is accelerating; extrapolation|0.99|
|MP2 📏 Galton's Heights|934 children, 1886|Regression to the mean; gender as a hidden variable|0.32|
|MP3 👶 Birth Weight|1,174 births (CHDS)|Smokers vs non-smokers at the same gestation; outliers|0.41|
|MP4 🐧 Palmer Penguins|342 penguins, 3 species|Overall *r* > every within-species *r*|0.87|
|MP5 ⚛️ Periodic Trends|83 elements|Outlier elements; tighter trend within a period; common cause|0.90|

**Several classes:** put one roster per class in this folder, named `class_<section>.csv` (e.g. `class_03.csv`, `class_05.csv`, `class_08.csv`). Each is read in the layout it was exported in, and each class is dealt out separately, so every class has all five projects. The gallery can show one class at a time.

## ⚙️ Setup

In [ ]:
import sys, os, glob, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "shared-public")))   # Fall 2026/shared-public (like Lab 8, one level deeper)
from init import *
import microproject_tools as mpt

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# Where everything lives (set in microproject_tools.py)
os.makedirs(mpt.SUBMIT_DIR, exist_ok=True)
try:
    os.chmod(mpt.MICRO_ROOT, 0o777); os.chmod(mpt.SUBMIT_DIR, 0o777)   # let every student write submissions
except PermissionError:
    pass
print('Assignments file :', mpt.ASSIGN_FILE)
print('Submissions folder:', mpt.SUBMIT_DIR)

projects = Table().with_columns(
    'project',  make_array(*mpt.PROJECTS.keys()),
    'title',    make_array(*[v[0] for v in mpt.PROJECTS.values()]),
    'notebook', make_array(*[v[1] for v in mpt.PROJECTS.values()]))
projects

## 1. Class Rosters
Upload one roster per class into this folder **on the hub**, named `class_<section>.csv`. The number in the file name becomes the class label (`class_08.csv` → `08`). Each file is read **in the layout it was exported in**:

|Layout|Example first row|
|---|---|
|Registrar export, no header|`"Ahmed, Anas",tut13755,8,,,,`|
|Canvas gradebook export|`Student,ID,SIS User ID,SIS Login ID,Section,…` (the *Points Possible* row and *Test Student* are skipped)|
|Simple table|`Last Name,First Name,Username` (any order; `First Name`/`Last Name`/`Name`, `Username`/`Login`/`Email`)|

The login column (AccessNet ID) lets each student's notebook find their assignment automatically; names are the fallback.

🔒 **Keep rosters off GitHub.** They hold student names and IDs, and the course repo is public. nbgitpuller never touches files that aren't in the repo, and this folder's `.gitignore` blocks `class_*.csv` from being committed. `class_sample.csv` (made-up students) is used only if no real roster is found.

In [ ]:
class_files = sorted(f for f in glob.glob('class_*.csv') if f != 'class_sample.csv')
# Or list them yourself:  class_files = ['class_03.csv', 'class_05.csv', 'class_08.csv']
if not class_files:
    class_files = ['class_sample.csv']
    print('No class rosters found here — using the made-up sample.')

rows = []
for f in class_files:
    one_class = mpt.read_roster(f)
    print(f'{f:18s} {len(one_class):3d} students   class label: {one_class[0]["section"]}')
    rows = rows + one_class

roster = Table().with_columns(
    'section',    make_array(*[r['section'] for r in rows]),
    'first_name', make_array(*[r['first'] for r in rows]),
    'last_name',  make_array(*[r['last'] for r in rows]))

# Display names are decided within each class: 'Justin C.' and 'Justin K.' only if they share a class
display_names = make_array()
for i in np.arange(roster.num_rows):
    classmates = roster.where('section', roster.column('section').item(i))
    shared = np.count_nonzero(classmates.column('first_name') == roster.column('first_name').item(i)) > 1
    first, last = roster.column('first_name').item(i), roster.column('last_name').item(i)
    display_names = np.append(display_names, f'{first} {last[:1].upper()}.' if shared and last else first)

roster = roster.with_columns(
    'display_name', display_names,
    'last_initial', make_array(*[str(s)[:1].upper() for s in roster.column('last_name')]),
    'user_key',     make_array(*[mpt.user_key(r['login']) for r in rows]))   # scrambled login, never the ID itself

missing_login = np.count_nonzero(roster.column('user_key') == '')
if missing_login:
    print(f'⚠️ {missing_login} students have no login in their roster; they will be matched by name.')
roster.group('section')

## 2. Random Assignment
Within each class, we shuffle the students with `np.random.choice(..., replace=False)` — a random **permutation** — and deal them out to the projects like cards, so the groups differ in size by at most one. The project order is shuffled too, so no project always gets the extra student.

Setting a **seed** makes the deal reproducible: rerun with the same seed and you get the same groups.

In [ ]:
def deal(class_roster, seed):
    "Randomly assign one class to the projects, in groups whose sizes differ by at most one."
    np.random.seed(seed)
    n = class_roster.num_rows
    shuffled = np.random.choice(np.arange(n), n, replace=False)             # a random order of the students
    project_order = np.random.choice(projects.column('project'), projects.num_rows, replace=False)
    dealt = project_order.take(np.arange(n) % projects.num_rows)           # deal like cards
    return class_roster.take(shuffled).with_columns('project', dealt)

seed = 1013
sections = np.unique(roster.column('section'))
dealt_classes = [deal(roster.where('section', sec), seed + i) for i, sec in enumerate(sections)]
new_assignments = dealt_classes[0]
for t in dealt_classes[1:]:
    new_assignments = new_assignments.append(t)
new_assignments = new_assignments.select('section', 'display_name', 'first_name', 'last_initial', 'user_key', 'project')
new_assignments.pivot('project', 'section')

### Save the assignments
Students' `my_project(name)` reads this file. Classes that are **already saved keep their groups**, so you can add a new class later without disturbing the others. To re-deal a class that students haven't seen yet, list it in `redeal`.

In [ ]:
redeal = make_array()          # e.g. make_array('08') to re-deal class 08

columns = new_assignments.labels
if os.path.exists(mpt.ASSIGN_FILE):
    saved = Table.read_table(mpt.ASSIGN_FILE, dtype=str).select(columns)
else:
    saved = new_assignments.where('section', 'no such class')          # an empty table with the same columns
saved_sections = np.unique(saved.column('section'))

to_write = make_array(*[s for s in sections if s not in saved_sections or s in redeal])
kept = saved.where('section', are.not_contained_in(to_write)) if len(to_write) else saved
added = new_assignments.where('section', are.contained_in(to_write)) if len(to_write) else new_assignments.where('section', 'no such class')
assignments = kept.append(added) if added.num_rows else kept

if len(to_write):
    assignments.to_csv(mpt.ASSIGN_FILE)
    try:
        os.chmod(mpt.ASSIGN_FILE, 0o644)
    except PermissionError:
        pass
print('Kept saved classes:   ', ', '.join(s for s in saved_sections if s not in to_write) or 'none')
print('Wrote new/re-dealt:   ', ', '.join(to_write) or 'none')
assignments.pivot('project', 'section')

Each group, ready to show on screen. Set `show_class` to one class label, or `None` for all classes.

In [ ]:
show_class = None          # e.g. '08'

for sec in np.unique(assignments.column('section')):
    if show_class is not None and sec != show_class:
        continue
    print(f'── Class {sec} ──')
    one_class = assignments.where('section', sec)
    for pid in projects.column('project'):
        names = one_class.where('project', pid).column('display_name')
        print(f"  {pid} — {mpt.PROJECTS[pid][0]:28s}  {', '.join(sorted(names))}")
    print()

## 3. Submission Gallery 🖼️
Rerun the cells below any time during class. Each submission is a PNG plus a small JSON file with the student's *r*, slope, intercept, caption, and class. Resubmissions replace older versions.

In [ ]:
def load_submissions():
    "Read every submission record into a Table."
    paths = sorted(glob.glob(os.path.join(mpt.SUBMIT_DIR, '*.json')))
    records = [json.load(open(p)) for p in paths]
    if not records:
        return Table().with_columns('project', make_array(), 'section', make_array(), 'name', make_array(),
                                    'first', make_array(), 'user_key', make_array(), 'png', make_array())
    return Table().with_columns(
        'project',   make_array(*[r['project'] for r in records]),
        'section',   make_array(*[str(r.get('section', '')) for r in records]),
        'name',      make_array(*[r['name'] for r in records]),
        'first',     make_array(*[r['first_name'].lower() for r in records]),
        'user_key',  make_array(*[r.get('user_key', '') for r in records]),
        'r',         make_array(*[r['r'] for r in records]),
        'slope',     make_array(*[r['slope'] for r in records]),
        'intercept', make_array(*[r['intercept'] for r in records]),
        'caption',   make_array(*[r['caption'] for r in records]),
        'submitted', make_array(*[r['submitted'] for r in records]),
        'warnings',  make_array(*[len(r.get('figure_warnings', [])) for r in records]),
        'png',       make_array(*[p[:-5] + '.png' for p in paths]))

submissions = load_submissions()
print(submissions.num_rows, 'submissions')
if submissions.num_rows:
    submissions.drop('png', 'first', 'user_key').sort('project').show()

### Who has submitted?

In [ ]:
show_class = None          # e.g. '08'

# Match on the hub login first; fall back to first name (only when it is unique in that class)
status = make_array()
for i in np.arange(assignments.num_rows):
    sec = assignments.column('section').item(i)
    first = str(assignments.column('first_name').item(i)).lower()
    same_class = assignments.where('section', sec)
    unique_first = np.count_nonzero(np.char.lower(same_class.column('first_name').astype(str)) == first) == 1
    class_subs = submissions.where('section', sec) if submissions.num_rows else submissions
    done = (assignments.column('user_key').item(i) in submissions.column('user_key')
            or (unique_first and first in class_subs.column('first')))
    status = np.append(status, '✅' if done else '—')
roster_status = assignments.with_columns('submitted', status)
if show_class is not None:
    roster_status = roster_status.where('section', show_class)

print(np.count_nonzero(roster_status.column('submitted') == '✅'), 'of', roster_status.num_rows, 'students have submitted')
roster_status.pivot('submitted', 'section').show()
roster_status.select('section', 'project', 'display_name', 'submitted').sort('display_name').sort('section').show()

### Show the figures

In [ ]:
def show_gallery(pid, section=None, cols=2):
    "Display every submitted figure for one project (optionally one class), with the student's caption."
    sub = submissions.where('project', pid)
    if section is not None:
        sub = sub.where('section', section)
    label = mpt.PROJECTS.get(pid, ('Instructor demo',))[0]
    print(f"{pid}: {label} — {sub.num_rows} figures" + (f' (class {section})' if section else ''))
    if sub.num_rows == 0:
        return
    rows = int(np.ceil(sub.num_rows / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 5.2 * rows), squeeze=False)
    for ax in axes.flatten():
        ax.axis('off')
    for i in np.arange(sub.num_rows):
        ax = axes.flatten()[i]
        ax.imshow(plt.imread(sub.column('png').item(i)))
        ax.set_title(sub.column('name').item(i), fontsize=14, weight='bold')
        ax.text(0.5, -0.04, sub.column('caption').item(i), transform=ax.transAxes,
                ha='center', va='top', fontsize=10, wrap=True)
    plt.tight_layout()
    plt.show()

show_gallery('MP1', section=show_class)

In [ ]:
for pid in projects.column('project'):
    show_gallery(pid, section=show_class)

## 4. Discussion: Did Everyone Get the Same Line?
Students on the same project used the same data, so their full-data *r* and slope **should agree**. Disagreement is a great discussion starter: a bug in someone's toolbox, a subset of the data (the twist line instead of the full-data line), or swapped $x$ and $y$.

Across projects, compare *r* with how good the fit *looked*: MP1 has the largest *r* but a clearly curved error plot; MP2 has the smallest *r* yet tells the oldest story in regression ([Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)).

In [ ]:
if submissions.num_rows:
    summary = submissions.select('project', 'r', 'slope').group('project', np.mean).join(
        'project', submissions.select('project', 'r').group('project', np.std).relabel('r std', 'spread of r'))
    summary.show()

**Prompts for the gallery walk**
- Which figure would you put in a newspaper? Why?
- Which project shows that a large *r* does not guarantee a straight line is the right model?
- In which projects did splitting the data into groups change the story?
- Which findings could you state as **causes**, and which only as **associations**?